# Week 10: Customer Churn Preprocessing & Feature Engineering

## Project Overview
This project focuses on preparing raw customer data for machine learning by building a complete data pipeline.

## Day 1: Explore & Understand
Loading the data and dropping non-informative columns like `CustomerID` early on.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Load data
df = pd.read_csv('churn_data.csv')
# Drop CustomerID immediately as it is just an identifier
df = df.drop('CustomerID', axis=1)

print(df.head())
print(df.info())

   Tenure  MonthlyCharges  TotalCharges        Contract     PaymentMethod  \
0       6              64          1540        One year       Credit Card   
1      21             113          1753  Month-to-month  Electronic Check   
2      27              31          1455        Two year       Credit Card   
3      53              29          7150  Month-to-month  Electronic Check   
4      16             185          1023        One year  Electronic Check   

  PaperlessBilling  SeniorCitizen  Churn  
0               No              1      0  
1              Yes              1      0  
2               No              1      0  
3               No              1      0  
4               No              1      0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Tenure            500 non-null    int64 
 1   MonthlyCharges    500 non-null  

## Day 2 & 3: Handle Categorical Data & Scaling
Exploring different encoding and scaling methods.

In [2]:
# Categorical Encoding Comparison
print("Contract unique values:", df['Contract'].unique())

# Label Encoding for PaperlessBilling
le = LabelEncoder()
paperless_enc = le.fit_transform(df['PaperlessBilling'])

# Scaling Comparison
std_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()
scaled_std = std_scaler.fit_transform(df[['MonthlyCharges', 'TotalCharges']])
scaled_mm = minmax_scaler.fit_transform(df[['MonthlyCharges', 'TotalCharges']])
print("Scaling models fit successfully.")

Contract unique values: ['One year' 'Month-to-month' 'Two year']
Scaling models fit successfully.


## Day 4: Outlier Detection
Using Z-score and IQR methods.

In [3]:
from scipy import stats

# Z-score method
z_scores = stats.zscore(df[['MonthlyCharges', 'TotalCharges']])
abs_z_scores = np.abs(z_scores)
outliers_z = (abs_z_scores > 3).all(axis=1)
print(f"Outliers detected via Z-score: {outliers_z.sum()}")

# IQR method
Q1 = df['MonthlyCharges'].quantile(0.25)
Q3 = df['MonthlyCharges'].quantile(0.75)
IQR = Q3 - Q1
print(f"MonthlyCharges IQR: {IQR}")

Outliers detected via Z-score: 0
MonthlyCharges IQR: 91.0


## Day 5: Feature Engineering
Creating new combined features.

In [4]:
df_eng = df.copy()
df_eng['Tenure_Ratio'] = df_eng['Tenure'] / 72.0 # Max tenure approx 72
df_eng['AverageMonthly'] = df_eng['TotalCharges'] / (df_eng['Tenure'] + 1)
df_eng['IsSenior_MonthToMonth'] = ((df_eng['SeniorCitizen'] == 1) & (df_eng['Contract'] == 'Month-to-month')).astype(int)
df_eng['Charge_Diff'] = df_eng['MonthlyCharges'] - df_eng['AverageMonthly']
df_eng['HighCharge_ShortTenure'] = ((df_eng['MonthlyCharges'] > 100) & (df_eng['Tenure'] < 12)).astype(int)

print(df_eng[['Tenure_Ratio', 'AverageMonthly', 'IsSenior_MonthToMonth', 'Charge_Diff', 'HighCharge_ShortTenure']].head())

   Tenure_Ratio  AverageMonthly  IsSenior_MonthToMonth  Charge_Diff  \
0      0.083333      220.000000                      0  -156.000000   
1      0.291667       79.681818                      1    33.318182   
2      0.375000       51.964286                      0   -20.964286   
3      0.736111      132.407407                      1  -103.407407   
4      0.222222       60.176471                      0   124.823529   

   HighCharge_ShortTenure  
0                       0  
1                       0  
2                       0  
3                       0  
4                       0  


## Day 7: Build Complete Pipeline
Automating the workflow.

In [5]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer

numeric_features = ['Tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = ['Contract', 'PaymentMethod', 'PaperlessBilling']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', RandomForestClassifier(random_state=42))])

X = df.drop('Churn', axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print("Pre-processing Pipeline Evaluation:")
print(classification_report(y_test, y_pred))

Pre-processing Pipeline Evaluation:
              precision    recall  f1-score   support

           0       0.94      1.00      0.97        84
           1       1.00      0.69      0.81        16

    accuracy                           0.95       100
   macro avg       0.97      0.84      0.89       100
weighted avg       0.95      0.95      0.95       100

